# Import Lib

# Online Retail project

## Part 1: Data Cleaning, Formatting & Univariate Analysis

In [26]:
# STEP 1: IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import plotly.io as pio 
# set the default temp to dark themed
pio.templates.default = 'plotly_dark'
import streamlit as st

# STEP 2: LOAD DATA & OVERVIEW

In [3]:
# Read the dataset from Kaggle CSV file
df = pd.read_csv('OnlineRetail.csv', encoding='ISO-8859-1')

In [4]:
# Show initial shape of the dataset
print(f"Dataset Shape (Rows, Columns): {df.shape}")

Dataset Shape (Rows, Columns): (541909, 8)


In [5]:
# Inspect first 5 rows of raw data
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [6]:
# Inspect column data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 67.3 MB


In [7]:
# Summary statistics for numerical variables
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [8]:
# Summary statistics for categorical variables
print("\n--- DESCRIPTIVE STATISTICS (CATEGORICAL) ---")
print(df.describe(include=['O']))


--- DESCRIPTIVE STATISTICS (CATEGORICAL) ---
       InvoiceNo StockCode                         Description  \
count     541909    541909                              540455   
unique     25900      4070                                4223   
top       573585    85123A  WHITE HANGING HEART T-LIGHT HOLDER   
freq        1114      2313                                2369   

             InvoiceDate         Country  
count             541909          541909  
unique             23260              38  
top     10/31/2011 14:41  United Kingdom  
freq                1114          495478  


C:\Users\moata\AppData\Local\Temp\ipykernel_30780\1006593612.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include=['O']))


## STEP 3: DATA CLEANING & ERROR HANDLING

In [9]:
"""
ERRORS IDENTIFIED AND HANDLED:
--------------------------------------------------------------------------------
1. Missing Values: 'CustomerID' contains ~135k null values. Missing CustomerIDs
   cannot be attributed to registered buyers, so we filter them into a cleaned dataset.
2. Cancelled Orders: Invoices starting with 'C' indicate cancellations with 
   negative 'Quantity' values. We isolate cancelled vs normal transactions.
3. Unit Price Anomalies: 'UnitPrice' contains zero or negative entries 
   (system adjustments/bad debts). Filter for UnitPrice > 0.
4. Data Types: 'InvoiceDate' is loaded as object/string -> convert to datetime.
   'CustomerID' is loaded as float -> convert to integer/string.
--------------------------------------------------------------------------------
"""

"\nERRORS IDENTIFIED AND HANDLED:\n--------------------------------------------------------------------------------\n1. Missing Values: 'CustomerID' contains ~135k null values. Missing CustomerIDs\n   cannot be attributed to registered buyers, so we filter them into a cleaned dataset.\n2. Cancelled Orders: Invoices starting with 'C' indicate cancellations with \n   negative 'Quantity' values. We isolate cancelled vs normal transactions.\n3. Unit Price Anomalies: 'UnitPrice' contains zero or negative entries \n   (system adjustments/bad debts). Filter for UnitPrice > 0.\n4. Data Types: 'InvoiceDate' is loaded as object/string -> convert to datetime.\n   'CustomerID' is loaded as float -> convert to integer/string.\n--------------------------------------------------------------------------------\n"

In [10]:
# Check for missing values count per column
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [11]:
# Clean Step A: Convert InvoiceDate to datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [12]:
# Clean Step B: Identify and create flag for cancelled orders (Invoice Starts with 'C')
df['IsCancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')

In [13]:
# Clean Step C: Remove rows with missing CustomerID for transaction-level analysis
df_clean = df.dropna(subset=['CustomerID']).copy()
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)
df['CustomerID'] = df['CustomerID'].fillna(0).astype(int)

In [14]:
# Clean Step D: Remove records with UnitPrice <= 0
df_clean = df_clean[df_clean['UnitPrice'] > 0]

In [15]:
# Clean Step E: Separate completed purchases from cancellations
df_sales = df_clean[df_clean['Quantity'] > 0].copy()

### Feature Engineering

In [16]:
# Feature Engineering: Add TotalAmount column (Quantity * UnitPrice)
df_sales['TotalAmount'] = df_sales['Quantity'] * df_sales['UnitPrice']

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  str           
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   541909 non-null  int64         
 7   Country      541909 non-null  str           
 8   IsCancelled  541909 non-null  bool          
dtypes: bool(1), datetime64[us](1), float64(1), int64(2), str(4)
memory usage: 60.0 MB


In [18]:
# Feature Engineering: Extract date parts for temporal analysis
df_sales['Year'] = df_sales['InvoiceDate'].dt.year
df_sales['Month'] = df_sales['InvoiceDate'].dt.month
df_sales['MonthName'] = df_sales['InvoiceDate'].dt.strftime('%b')
df_sales['DayOfWeek'] = df_sales['InvoiceDate'].dt.day_name()
df_sales['Hour'] = df_sales['InvoiceDate'].dt.hour
df_sales['YearMonth'] = df_sales['InvoiceDate'].dt.to_period('M').astype(str)

In [19]:
# Display final summary after cleaning
print(f"\nCleaned Dataset Shape: {df_sales.shape}")


Cleaned Dataset Shape: (397884, 16)


In [20]:
# Save cleaned dataset to CSV for Part 2
df_sales.to_csv('online_retail_cleaned_2.csv', index=False)
print("Cleaned data successfully saved to 'online_retail_cleaned.csv'")

Cleaned data successfully saved to 'online_retail_cleaned.csv'


## STEP 4: UNIVARIATE ANALYSIS (PLOTLY VISUALS)

In [27]:
# 1. Distribution of Quantity Sold
fig_qty = px.histogram(
    df_sales[df_sales['Quantity'] < 100], 
    x='Quantity', 
    nbins=50, 
    title='Univariate Analysis: Distribution of Order Quantities (< 100)',
    color_discrete_sequence=['#00CC96']
)
fig_qty.show()

In [28]:
# 2. Distribution of Unit Price
fig_price = px.histogram(
    df_sales[df_sales['UnitPrice'] < 20], 
    x='UnitPrice', 
    nbins=50, 
    title='Univariate Analysis: Distribution of Unit Price (< $20)',
    color_discrete_sequence=['#AB63FA']
)
fig_price.show()

In [29]:
# 3. Distribution of Total Amount per Line Item
fig_total = px.histogram(
    df_sales[df_sales['TotalAmount'] < 200], 
    x='TotalAmount', 
    nbins=50, 
    title='Univariate Analysis: Distribution of Line Item Total Revenue (< $200)',
    color_discrete_sequence=['#FFA15A']
)
fig_total.show()

In [30]:
# 4. Top 10 Countries by Transaction Count
country_counts = df_sales['Country'].value_counts().head(10).reset_index()
country_counts.columns = ['Country', 'TransactionCount']
fig_country = px.bar(
    country_counts, 
    x='Country', 
    y='TransactionCount',
    title='Univariate Analysis: Top 10 Countries by Transaction Volume',
    color='TransactionCount',
    color_continuous_scale='Turbo'
)
fig_country.show()